# Implement FactSales fact table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_orders = spark.table("silver.orders")

In [0]:
df_orders.limit(5).display()

In [0]:
df_order_items = spark.table("silver.order_items")

In [0]:
df_order_items.limit(5).display()

In [0]:
df_order_items.filter(F.col("order_item_id") != 1).limit(5).display()

In [0]:
df_order_items.groupBy("order_id").agg(F.count("order_item_id").alias("count")).orderBy(F.desc("count")).display()

In [0]:
df_order_items = df_order_items.select(["order_id", "order_item_id", "product_id", "price", "freight_value"])

In [0]:
df_order_items.limit(5).display()

In [0]:
df_products = spark.table("gold.dim_product")
df_products.limit(2).display()

In [0]:
seed = df_order_items.join(
    df_products.select("product_id", "product_sk"),
    "product_id",
    "left"
).drop("product_id")

In [0]:
seed.limit(5).display()

In [0]:
df_customers = spark.table("gold.dim_customer")
df_customers.limit(2).display()

In [0]:
spark.table("silver.customers").filter(F.col("customer_id") == "3ce436f183e68e07877b285a838db11a").display()

## Now its time for connection customers into factSales. Follow the below flow

```
order_items
  → orders            (gets customer_id)
  → silver.customers  (customer_id → customer_unique_id)   ← THE BRIDGE
  → dim_customer      (customer_unique_id → customer_sk, point-in-time)
```

In [0]:
silver_customers = spark.table("silver.customers")
silver_orders = spark.table("silver.orders")
gold_customers = spark.table("gold.dim_customer")
seed_orders = seed


In [0]:
seed.limit(1).display()

In [0]:
seed = seed.join(
    silver_orders.select("order_id", "customer_id"),
    on="order_id",
    how="left"
)


In [0]:
seed.limit(1).display()

In [0]:
seed = seed.join(silver_customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left").drop("customer_id")

In [0]:
seed.limit(2).display()

In [0]:
seed = seed.join(
    gold_customers.select("customer_unique_id", "customer_sk", "effective_date", "end_date"),
    (seed.customer_unique_id == gold_customers.customer_unique_id) &
    (seed.order_date >= gold_customers.effective_date) &
    (seed.order_date <= gold_customers.end_date),
    "left"
)

In [0]:
seed.limit(2).display()